# 18i — Corridors-out isolation test (PF-4, study plan v0.18.1): the one certified anchor solve

Reads `spec/v3.1/pf4_corridors_out.json` (written by 18h): the registered balanced formulation (S0 at SSP585, identity
shapes, manifest v3.1 targets) with climate corridors at weight 0 and transboundary connectivity carrying the connectivity block's
full 25% share; weights re-derived under constant intended influence for the other blocks. One anchor at opt_gap 1e-4 (Gurobi,
binary, NumericFocus; live internet) into `runs_v3.1/pf4_corridors_out/s0_ssp585_theta5/` — `anchor.tif` + `formulation_meta.json`.
No guarded sweep, no MGA; ~1 min. Resumable (skips if the anchor exists). Then 18j (the comparison and the reading rule).


In [1]:
ANALYSIS <- "y2y"
PROJ <- normalizePath(getwd())
while (!file.exists(file.path(PROJ, "config.py"))) {
  parent <- dirname(PROJ)
  if (identical(parent, PROJ)) stop("config.py not found above getwd() -- open from inside the repo")
  PROJ <- parent
}
setwd(PROJ)
source(file.path(PROJ, "prioritizr_core.R"))
source(file.path(PROJ, "mga_core.R"))
mpath <- pr_refresh_manifest(PROJ, ANALYSIS)
VERSION <- "v3.1"
MANIFEST_REL <- sprintf("analyses/y2y/spec/manifest_%s.csv", VERSION); FREEZE_REL <- sprintf("analyses/y2y/spec/manifest_%s.sha256", VERSION)
RUNS_REL <- sprintf("analyses/y2y/runs_%s", VERSION); EFG_SUBDIR_EXPECTED <- paste0("iucn_efg_", sub("\\..*$", "", VERSION))
MAN <- read.csv(file.path(PROJ, MANIFEST_REL), stringsAsFactors = FALSE)
dig <- strsplit(readLines(file.path(PROJ, FREEZE_REL))[1], "  ")[[1]][1]
stopifnot(identical(unname(tools::sha256sum(file.path(PROJ, MANIFEST_REL))[[1]]), dig), nrow(MAN) == 12)
PF4 <- jsonlite::read_json(file.path(PROJ, sprintf("analyses/y2y/spec/%s/pf4_corridors_out.json", VERSION)))
row <- MAN[MAN$formulation_id == PF4$base_formulation, ]; stopifnot(nrow(row) == 1)
w <- unlist(PF4$weights); t <- unlist(PF4$targets)
stopifnot(w[["climate_corridors"]] == 0, abs(w[["transboundary_connectivity"]] / unlist(PF4$weights_registered)[["transboundary_connectivity"]] - 1) > 0.5)
cat(sprintf("PF-4 corridors-out: %s | transboundary weight %.4f (registered %.4f) | corridors weight %.1f | opt_gap %g\n",
            PF4$base_formulation, w[["transboundary_connectivity"]], unlist(PF4$weights_registered)[["transboundary_connectivity"]], w[["climate_corridors"]], as.numeric(PF4$opt_gap)))


manifest refreshed from config.py (analysis=y2y)
PF-4 corridors-out: s0_ssp585_theta5 | transboundary weight 1.2361 (registered 0.6693) | corridors weight 0.0 | opt_gap 0.0001


In [2]:
# ---- the one certified anchor (mirrors 18f's anchor step; no MGA) ---------------------------------------------------------
OUT_REL <- file.path(RUNS_REL, PF4$arm, PF4$base_formulation); OUT <- file.path(PROJ, OUT_REL); dir.create(OUT, recursive = TRUE, showWarnings = FALSE)
if (file.exists(file.path(OUT, "anchor.tif"))) {
  cat(sprintf("%s: anchor exists -- skipped (delete the folder to re-solve)\n", OUT_REL))
} else {
  ctx <- pr_setup(mpath, PROJ)
  ctx <- modifyList(ctx, pr_ingest(ctx)); ctx <- modifyList(ctx, pr_planning_units(ctx))
  stopifnot(all(grepl(paste0("/", EFG_SUBDIR_EXPECTED, "/"), ctx$layers$path[ctx$layers$role == "feature_efg"])))
  actx <- pr_override(ctx, targets = as.list(t), feature_weight_multipliers = as.list(w),
                      results_dir = OUT_REL, results_subdir = "anchor_build",
                      solver = "gurobi", decision_type = "binary", opt_gap = as.numeric(PF4$opt_gap), portfolio_n = 1)
  actx <- modifyList(actx, pr_weights(actx)); actx <- modifyList(actx, pr_targets(actx)); actx <- modifyList(actx, pr_penalty_matrices(actx))
  bp <- pr_build_problem(actx); actx$p <- bp$p; actx$solve_params <- bp$solve_params
  cm <- mga_compile(actx)
  t0 <- proc.time()[["elapsed"]]
  anchor <- mga_anchor(cm, opt_gap = as.numeric(PF4$opt_gap))
  r <- terra::rast(actx$cost); v <- rep(NA_integer_, terra::ncell(r)); v[cm$pu_index] <- as.integer(anchor$x); terra::values(r) <- v; names(r) <- "anchor_01"
  terra::writeRaster(r, file.path(OUT, "anchor.tif"), overwrite = TRUE, datatype = "INT1U", NAflag = 255, gdal = c("COMPRESS=DEFLATE", "TILED=YES"))
  jsonlite::write_json(list(formulation_id = PF4$base_formulation, kind = "anchor_only", variant = PF4$label, blocks = PF4$blocks,
                            anchor_objective = anchor$z, anchor_bound = anchor$bound, anchor_gap = anchor$gap, anchor_runtime_s = anchor$runtime,
                            wall_s = proc.time()[["elapsed"]] - t0, n_selected = sum(anchor$x), weights = as.list(w), targets = as.list(t),
                            created_utc = format(Sys.time(), tz = "UTC")),
                       file.path(OUT, "formulation_meta.json"), auto_unbox = TRUE, pretty = TRUE, digits = 10)
  cat(sprintf("PF-4 anchor: z %.6f (bound %.6f, gap %.1e) | %d cells | %.0f s -> %s\n", anchor$z, anchor$bound, anchor$gap, sum(anchor$x), anchor$runtime, OUT_REL))
}
cat("next: 18j_pf4_corridors_out_analysis\n")


prioritizr 8.1.0 | terra 1.9.34 | analysis=y2y | solver=highs (single solution)
objective=min_shortfall | budget=30% target=100% | opt_gap=0.10 | time_limit=43200s
resolution: 1000 m (agg factor 1) | decisions=proportion
roi: mode=full | lock_in=pa_mask
penalties: connectivity=0 | boundary=0 | neighbor=0
outputs -> output_data/iter6_y2y
ingested 28 features (8 continuous + 20 EFG) + cost + PA mask | grid 1286 x 3312 @ 1000 m
normalized 28 features to total=100000 each (scale-invariant conditioning)
planning units: 1,272,914 cells | budget = 30% = 381,874 cells
locked-in [pa_mask]: 191,029 cells (15.0% of window) -- fits within budget
  override targets          -> irrecoverable_carbon_m_soc=0.332, F1.1.web.mix_v2.0=0.5013, F2.1.web.alt_v4.0=0.4912, T6.1.web.map_v1.0=0.3588, F1.2.web.map_v1.0=0.2451, T2.2.web.mix_v1.0=0.1476, T3.4.web.mix_v1.0=0.1094, T5.4.web.mix_v1.0=0.1046, F1.6.web.mix_v1.0=0.1009, T4.4.web.orig_v1.0=0.1, SF1.2.web.orig_v1.0=0.1, T6.2.web.alt_v2.0=0.1, T6.3.web.map_

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (28 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.1 and 1)
││└•weights:    continuous values (between 0 and 1.576664)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



compiled: 1272942 cols (1272914 pu + 28 aux) x 29 rows | 191029 locked pu | modelsense min
anchor: objective 4.182620 (bound 4.182612, gap 2.03e-06) | 381,874 selected | 41 s
PF-4 anchor: z 4.182620 (bound 4.182612, gap 2.0e-06) | 381874 cells | 39 s -> analyses/y2y/runs_v3.1/pf4_corridors_out/s0_ssp585_theta5
next: 18j_pf4_corridors_out_analysis
